In [ ]:
# ==============================================================
# 【AI 赛道】AI 应用说明
# AI 技术类型: LLM (CodeBuddy CLI)
# AI 参与环节: 因子构想生成、代码实现、自动调试
# ==============================================================
#
# factor_1258 — MOAP-DVA-MW (Morning-Open Absorption + Multi-Window VWAP Anchor)
# 早盘放量吸筹 × 多窗口动态VWAP锚定价格修复 反转确认因子 (突变自 factor_1082)
#
# 核心逻辑 (concrete 模式):
#   在父代 MOAP-DVA (单一5日VWAP锚) 基础上，引入**多窗口动态VWAP锚定组合**：
#   收盘价相对 3日/5日/10日 三窗口滚动VWAP 的加权偏离作为价格修复锚。
#   早盘放量吸筹 + 收盘站上多窗口VWAP(跨日持续吸筹) + 尾盘平稳承接 + 波动环境阻尼，
#   构成更稳健的"开盘大资金进，多日被买回"吸筹模式。
#
# 对父代的有意义改动:
#   1. 引入 3日/5日/10日 三窗口滚动VWAP锚 (0.35/0.4/0.25 加权) — 替代单一5日锚，
#      同时捕捉短期(3日)修复与中期(10日)趋势确认，跨日持续吸筹信号更强 (窗口扩展)
#   2. 早盘量能占比改用加权 + 5日平滑 (m_sum 平滑吸筹持续性)，替代单日瞬时占比，
#      降低单日噪声 (平滑/窗口调整)
#   3. 尾盘承接用量能集中度倒数 (1 - 最大单分钟占比) 乘法加强"平稳持续承接"过滤
#      (条件加强)，并加入 5日尾盘占比平滑
#   4. 波动环境阻尼: 用 (high-low)/vwap 归一化 + 更陡 tanh 压缩过滤极端行情，
#      并加入 5日滚动波动率门控 (1 - tanh(vol5/std))
#
# 因子方向:
#   因子值大 = 早盘放量吸筹 + 收盘站上多窗口VWAP(多日修复) + 尾盘平稳承接
#   → 预测次日继续上涨
#

def main(datasources, start_date, end_date):
    """
    因子构建主函数 - Morning-Open Absorption + Multi-Window VWAP Anchor (MOAP-DVA-MW)

    参数:
        datasources (dict): {"bar1m": "表名", "financial": "表名"}
        start_date (str): 开始时间
        end_date (str): 结束时间

    返回:
        pd.DataFrame: [date, instrument, factor]
    """
    import pandas as pd
    import dai

    # 1. 取数据源表名
    bar1m_table = datasources["bar1m"]

    # 2. 缓冲期 (10日滚动VWAP + 5日平滑需要回溯约 16 个交易日 + 余量)
    LOOKBACK_DAYS = 26
    query_start = pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)

    # 3. DAI SQL 计算因子
    sql = f"""
    WITH intraday AS (
        SELECT 
            date,
            instrument,
            close,
            high,
            low,
            amount,
            volume,
            CAST(strftime(date, '%H%M') AS INTEGER) AS time_int
        FROM {bar1m_table}
    ),
    daily_stats AS (
        SELECT 
            date::DATE::DATETIME AS date,
            instrument,
            max(high) AS day_high,
            min(low) AS day_low,
            last(close ORDER BY date) AS close_price,
            sum(amount) AS day_amount,
            sum(volume) AS day_volume,
            sum(amount) / NULLIF(sum(volume), 0) AS vwap,
            -- 早盘(09:30-10:00)成交占比
            sum(CASE WHEN time_int >= 930 AND time_int < 1000 THEN volume ELSE 0 END)
                / NULLIF(sum(volume), 0) AS am_vol_ratio,
            -- 尾盘(>=1430)成交占比
            sum(CASE WHEN time_int >= 1430 THEN volume ELSE 0 END)
                / NULLIF(sum(volume), 0) AS tail_vol_ratio,
            -- 尾盘量能集中度: 最大单分钟占比 (用于平稳承接过滤)
            max(CASE WHEN time_int >= 1430 THEN volume ELSE 0 END)
                / NULLIF(sum(volume), 0) AS tail_max_ratio
        FROM intraday
        GROUP BY date::DATE, instrument
    ),
    anchored AS (
        SELECT 
            date,
            instrument,
            day_high,
            day_low,
            close_price,
            vwap,
            am_vol_ratio,
            tail_vol_ratio,
            tail_max_ratio,
            -- 收盘相对当日VWAP修复 (父代核心)
            close_price / NULLIF(vwap, 0) - 1.0 AS close_rel_vwap,
            -- 多窗口滚动VWAP偏离: 3日/5日/10日
            close_price / NULLIF(
                m_sum(day_amount, 3) / NULLIF(m_sum(day_volume, 3), 0), 0
            ) - 1.0 AS close_rel_vwap3,
            close_price / NULLIF(
                m_sum(day_amount, 5) / NULLIF(m_sum(day_volume, 5), 0), 0
            ) - 1.0 AS close_rel_vwap5,
            close_price / NULLIF(
                m_sum(day_amount, 10) / NULLIF(m_sum(day_volume, 10), 0), 0
            ) - 1.0 AS close_rel_vwap10,
            -- 早盘量能占比 5日平滑 (吸筹持续性)
            avg(am_vol_ratio) OVER (PARTITION BY instrument ORDER BY date
                ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS am_vol_ratio5,
            -- 尾盘量能占比 5日平滑 (承接持续性)
            avg(tail_vol_ratio) OVER (PARTITION BY instrument ORDER BY date
                ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS tail_vol_ratio5,
            -- 5日滚动波动率 (相对环境门控)
            nanstd(day_high / NULLIF(day_low, 0) - 1.0) OVER (
                PARTITION BY instrument ORDER BY date
                ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS vol5,
            -- 日内振幅 (相对VWAP归一化)
            (day_high - day_low) / NULLIF(vwap, 0) AS day_range
        FROM daily_stats
    )
    SELECT 
        date,
        instrument,
        c_rank(tanh(
            -- 早盘放量吸筹 (平滑 + sqrt压缩, 加法项)
            sqrt(NULLIF(am_vol_ratio5, 0))
            -- 当日VWAP修复 × 多窗口VWAP修复 (双锚价格修复, 乘法组合)
            * (1.0 + tanh(close_rel_vwap * 2.0))
            * (1.0
                + tanh(close_rel_vwap3 * 1.5) * 0.35
                + tanh(close_rel_vwap5 * 2.0) * 0.40
                + tanh(close_rel_vwap10 * 1.5) * 0.25)
            -- 尾盘平稳承接: 量能持续而非单分钟脉冲 (平滑 + 集中度倒数)
            * (1.0 + LOG(1.0 + NULLIF(tail_vol_ratio5, 0) * 3.0)
                * (1.0 - NULLIF(tail_max_ratio, 0) * 3.0))
            -- 温和振幅门控: 过滤极端行情噪声
            * (1.0 - tanh(NULLIF(day_range, 0) * 2.0))
            -- 波动环境门控: 低波环境强化 (5日滚动波动率收缩)
            * (1.0 - tanh(NULLIF(vol5, 0) * 5.0))
        )) AS factor
    FROM anchored
    """

    df = dai.query(
        sql, filters={'date': [query_start, end_date]}, compression=True
    ).df()

    # 4. 对齐中证1000成分股
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()
    df = pd.merge(df, stk_pool, how='inner', on=['date', 'instrument'])

    # 5. 处理极值和缺失值
    df['factor'] = df['factor'].replace([float('inf'), float('-inf')], None)
    df = df.dropna(subset=['factor'])

    return df[['date', 'instrument', 'factor']]

## 本地测试


In [ ]:
if __name__== "__main__":
    from bigquant import dai
    from bigmodule import M
    import structlog
    logger = structlog.get_logger()

    datasources = {"bar1m": "bigalpha_2026_stock_bar1m", 'financial': 'bigalpha_2026_financial'}
    start_date = "2019-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"

    factor_data = main(datasources, start_date, end_date)
    logger.info("Factor computed", rows=len(factor_data))

    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={"date": ["2019-01-01", "2024-12-31"]},
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )